In [4]:
# EXERCISE 1
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "German"
STUDENT_ID = "TUPM-26-4336"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(int(d) for d in id_digits)
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
description_path = DATA_DIR / "secom.names"

for path in [measurement_path, label_path, description_path]:
    assert path.exists(), f"Missing file: {path}"

data = np.genfromtxt(measurement_path, dtype=float)

print("=== EXERCISE 1: MEASUREMENT QUALITY ANALYSIS ===")
print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Random Seed:", RANDOM_SEED)
print("Dataset Shape:", data.shape)

missing = np.isnan(data)
observed = (~missing).sum(axis=0)

all_missing = observed == 0

minimum = np.where(missing, np.inf, data).min(axis=0)
maximum = np.where(missing, -np.inf, data).max(axis=0)

constant = (observed > 0) & (minimum == maximum)
usable = ~(all_missing | constant)

usable_indices = np.flatnonzero(usable)

print("\nTotal Features:", data.shape[1])
print("Entirely Missing Features:", all_missing.sum())
print("Constant Features:", constant.sum())
print("Usable Features:", len(usable_indices))

number_to_select = 5 + (SEED_DIGIT % 3)

rng = np.random.default_rng(RANDOM_SEED)

selected_indices = rng.choice(
    usable_indices,
    size=number_to_select,
    replace=False
)

selected_indices = np.sort(selected_indices)

feature_names = [
    f"Feature_{i:03d}" for i in selected_indices
]

selected_data = data[:, selected_indices]

feature_means = np.nanmean(selected_data, axis=0)
feature_stds = np.nanstd(selected_data, axis=0, ddof=1)

missing_percentages = (
    np.isnan(selected_data).mean(axis=0) * 100
)

medians = np.nanmedian(selected_data, axis=0)

mads = np.nanmedian(
    np.abs(selected_data - medians),
    axis=0
)

robust_values = np.zeros_like(selected_data)

np.divide(
    0.6745 * (selected_data - medians),
    mads,
    out=robust_values,
    where=(mads > 0)
)

zero_mad = mads == 0

robust_values = np.where(
    zero_mad & (selected_data != medians),
    np.inf,
    robust_values
)

robust_values = np.where(
    np.isnan(selected_data),
    np.nan,
    robust_values
)

flags = (
    (np.abs(robust_values) > 3.5)
    & (~np.isnan(selected_data))
)

flagged_counts = flags.sum(axis=0)

observed_counts = (~np.isnan(selected_data)).sum(axis=0)

flagged_percentages = (
    flagged_counts / observed_counts * 100
)

quality_summary = pd.DataFrame({
    "Feature": feature_names,
    "Mean": feature_means,
    "Sample_SD": feature_stds,
    "Missing_Percentage": missing_percentages,
    "Median": medians,
    "MAD": mads,
    "Flagged_Count": flagged_counts,
    "Flagged_Percentage": flagged_percentages
})

quality_summary = quality_summary.sort_values(
    "Flagged_Percentage",
    ascending=False
)

output_path = OUTPUT_DIR / "measurement_quality.csv"

quality_summary.to_csv(output_path, index=False)

loaded_summary = pd.read_csv(output_path)

assert len(loaded_summary) == number_to_select
assert loaded_summary["Feature"].notna().all()

print("\nSELECTED FEATURES:")
for feature in feature_names:
    print(feature)

print("\nMEASUREMENT QUALITY SUMMARY:")
print(quality_summary.round(4).to_string(index=False))

print("\nASSESSMENT DATA")
print("Selected Features:", len(feature_names))
print(
    "Highest Missing Percentage:",
    f"{missing_percentages.max():.4f}%"
)
print(
    "Feature with Highest Flagged Percentage:",
    feature_names[np.argmax(flagged_percentages)]
)
print("Total Flagged Observations:", flags.sum())
print("Vectorized Results Verified:", True)

print("\nCSV FILE")
print("Filename:", output_path)
print("File Created:", output_path.exists())

=== EXERCISE 1: MEASUREMENT QUALITY ANALYSIS ===
Student: German
Student ID: TUPM-26-4336
Random Seed: 264336
Dataset Shape: (1567, 590)

Total Features: 590
Entirely Missing Features: 0
Constant Features: 116
Usable Features: 474

SELECTED FEATURES:
Feature_054
Feature_119
Feature_250
Feature_349
Feature_523

MEASUREMENT QUALITY SUMMARY:
    Feature     Mean  Sample_SD  Missing_Percentage   Median     MAD  Flagged_Count  Flagged_Percentage
Feature_523   0.4539     4.1476              0.0000   0.1000  0.0288            192             12.2527
Feature_349   0.0253     0.0106              1.5316   0.0240  0.0045             41              2.6572
Feature_054   4.8385     0.0596              0.2553   4.8430  0.0260             11              0.7038
Feature_250 109.6510    54.5973              0.0000 103.0936 27.8314              8              0.5105
Feature_119   0.9708     0.0089              0.0000   0.9694  0.0064              3              0.1914

ASSESSMENT DATA
Selected Features:

In [5]:
# EXERCISE 2
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "German"
STUDENT_ID = "TUPM-26-4336"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(int(d) for d in id_digits)
RANDOM_SEED = int(id_digits) % (2**32)

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
description_path = DATA_DIR / "secom.names"

for path in [measurement_path, label_path, description_path]:
    assert path.exists(), f"Missing file: {path}"

measurements = np.genfromtxt(measurement_path, dtype=float)
labels = np.loadtxt(label_path, usecols=0, dtype=int)

print("=== EXERCISE 2: PASS/FAIL COMPARISON ===")
print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Random Seed:", RANDOM_SEED)

missing = np.isnan(measurements)
observed = (~missing).sum(axis=0)

all_missing = observed == 0

minimum = np.where(missing, np.inf, measurements).min(axis=0)
maximum = np.where(missing, -np.inf, measurements).max(axis=0)

constant = (observed > 0) & (minimum == maximum)
usable = ~(all_missing | constant)

usable_indices = np.flatnonzero(usable)

pass_indices = np.flatnonzero(labels == -1)
fail_indices = np.flatnonzero(labels == 1)

pass_count = len(pass_indices)
fail_count = len(fail_indices)

rng = np.random.default_rng(RANDOM_SEED)

selected_indices = rng.choice(
    usable_indices,
    size=5,
    replace=False
)

selected_indices = np.sort(selected_indices)

feature_names = [
    f"Feature_{i:03d}" for i in selected_indices
]

sample_per_group = min(
    20 + ID_SUM,
    pass_count,
    fail_count
)

selected_pass = rng.choice(
    pass_indices,
    size=sample_per_group,
    replace=False
)

selected_fail = rng.choice(
    fail_indices,
    size=sample_per_group,
    replace=False
)

selected_rows = np.concatenate([
    selected_pass,
    selected_fail
])

feature_medians = np.nanmedian(
    measurements[:, selected_indices],
    axis=0
)

sample_data = measurements[
    selected_rows
][:, selected_indices]

sample_data = np.where(
    np.isnan(sample_data),
    feature_medians,
    sample_data
)

sample_labels = labels[selected_rows]

df = pd.DataFrame(
    sample_data,
    columns=feature_names
)

df.insert(0, "Record_ID", selected_rows + 1)

df["Result"] = sample_labels

df["Outcome"] = np.where(
    sample_labels == -1,
    "Pass",
    "Fail"
)

print("\nDATASET INFORMATION")
print("Total Records:", len(measurements))
print("Pass Records:", pass_count)
print("Fail Records:", fail_count)
print("Balanced Sample per Group:", sample_per_group)
print("Total Sample Records:", len(df))

grouped = df.groupby("Outcome")[feature_names].agg(
    ["count", "mean", "median", "std"]
)

comparison_rows = []

numpy_pandas_match = True

for feature in feature_names:

    pass_values = df.loc[
        df["Outcome"] == "Pass", feature
    ].to_numpy()

    fail_values = df.loc[
        df["Outcome"] == "Fail", feature
    ].to_numpy()

    pass_mean = np.mean(pass_values)
    fail_mean = np.mean(fail_values)

    pass_median = np.median(pass_values)
    fail_median = np.median(fail_values)

    pass_sd = np.std(pass_values, ddof=1)
    fail_sd = np.std(fail_values, ddof=1)

    pandas_pass = grouped.loc["Pass", feature]
    pandas_fail = grouped.loc["Fail", feature]

    match = np.allclose(
        [
            pass_mean, fail_mean,
            pass_median, fail_median,
            pass_sd, fail_sd
        ],
        [
            pandas_pass["mean"],
            pandas_fail["mean"],
            pandas_pass["median"],
            pandas_fail["median"],
            pandas_pass["std"],
            pandas_fail["std"]
        ],
        rtol=1e-10,
        atol=1e-10
    )

    numpy_pandas_match = numpy_pandas_match and match

    mean_difference = abs(fail_mean - pass_mean)

    pooled_sd = np.sqrt(
        (
            (len(pass_values) - 1) * pass_sd**2
            +
            (len(fail_values) - 1) * fail_sd**2
        )
        /
        (
            len(pass_values) + len(fail_values) - 2
        )
    )

    if pooled_sd > 0:
        standardized_difference = mean_difference / pooled_sd
    elif mean_difference == 0:
        standardized_difference = 0
    else:
        standardized_difference = np.inf

    comparison_rows.append({
        "Feature": feature,
        "Pass_Count": len(pass_values),
        "Fail_Count": len(fail_values),
        "Pass_Mean": pass_mean,
        "Fail_Mean": fail_mean,
        "Pass_Median": pass_median,
        "Fail_Median": fail_median,
        "Pass_SD": pass_sd,
        "Fail_SD": fail_sd,
        "Absolute_Mean_Difference": mean_difference,
        "Pooled_SD": pooled_sd,
        "Standardized_Mean_Difference": standardized_difference
    })

comparison = pd.DataFrame(comparison_rows)

comparison = comparison.sort_values(
    "Standardized_Mean_Difference",
    ascending=False
)

sample_path = OUTPUT_DIR / "balanced_sample.csv"
comparison_path = OUTPUT_DIR / "feature_comparison.csv"

df.to_csv(sample_path, index=False)
comparison.to_csv(comparison_path, index=False)

loaded_sample = pd.read_csv(sample_path)

assert len(loaded_sample) == 2 * sample_per_group

assert (
    loaded_sample["Outcome"].value_counts()["Pass"]
    == sample_per_group
)

assert (
    loaded_sample["Outcome"].value_counts()["Fail"]
    == sample_per_group
)

print("\nSELECTED FEATURES:")
for feature in feature_names:
    print(feature)

print("\nFEATURE COMPARISON:")
print(comparison.round(4).to_string(index=False))

print("\nASSESSMENT DATA")
print("Balanced Sample Size:", sample_per_group, "per outcome")
print("Selected Features:", len(feature_names))
print(
    "Highest-Ranked Feature:",
    comparison.iloc[0]["Feature"]
)
print(
    "Absolute Standardized Mean Difference:",
    round(
        comparison.iloc[0]["Standardized_Mean_Difference"],
        4
    )
)
print("NumPy-Pandas Statistics Match:", numpy_pandas_match)

print("\nGENERATED FILES")
print("Balanced Sample:", sample_path.exists())
print("Feature Comparison:", comparison_path.exists())

=== EXERCISE 2: PASS/FAIL COMPARISON ===
Student: German
Student ID: TUPM-26-4336
Random Seed: 264336

DATASET INFORMATION
Total Records: 1567
Pass Records: 1463
Fail Records: 104
Balanced Sample per Group: 44
Total Sample Records: 88

SELECTED FEATURES:
Feature_054
Feature_119
Feature_250
Feature_349
Feature_523

FEATURE COMPARISON:
    Feature  Pass_Count  Fail_Count  Pass_Mean  Fail_Mean  Pass_Median  Fail_Median  Pass_SD  Fail_SD  Absolute_Mean_Difference  Pooled_SD  Standardized_Mean_Difference
Feature_250          44          44    99.2765   111.4798      94.3022     109.6430  33.1478  51.2088                   12.2034    43.1342                        0.2829
Feature_349          44          44     0.0249     0.0270       0.0234       0.0229   0.0110   0.0174                    0.0021     0.0146                        0.1411
Feature_523          44          44     0.2877     0.2163       0.0999       0.0818   0.6135   0.4007                    0.0713     0.5181                   

In [6]:
# EXERCISE 3
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "German"
STUDENT_ID = "TUPM-26-4336"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(int(d) for d in id_digits)
RANDOM_SEED = int(id_digits) % (2**32)

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
description_path = DATA_DIR / "secom.names"

for path in [measurement_path, label_path, description_path]:
    assert path.exists(), f"Missing file: {path}"

measurements = np.genfromtxt(measurement_path, dtype=float)
labels = np.loadtxt(label_path, usecols=0, dtype=int)

print("=== EXERCISE 3: ROBUSTNESS AND SENSITIVITY ===")
print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Random Seed:", RANDOM_SEED)

missing = np.isnan(measurements)
observed = (~missing).sum(axis=0)

all_missing = observed == 0

minimum = np.where(missing, np.inf, measurements).min(axis=0)
maximum = np.where(missing, -np.inf, measurements).max(axis=0)

constant = (observed > 0) & (minimum == maximum)
usable = ~(all_missing | constant)

usable_indices = np.flatnonzero(usable)

pass_indices = np.flatnonzero(labels == -1)
fail_indices = np.flatnonzero(labels == 1)

eligible_features = []

for feature_index in usable_indices:

    pass_observed = np.sum(
        ~np.isnan(measurements[pass_indices, feature_index])
    )

    fail_observed = np.sum(
        ~np.isnan(measurements[fail_indices, feature_index])
    )

    if pass_observed >= 10 and fail_observed >= 10:
        eligible_features.append(feature_index)

rng = np.random.default_rng(RANDOM_SEED)

selected_indices = rng.choice(
    eligible_features,
    size=5,
    replace=False
)

selected_indices = np.sort(selected_indices)

feature_names = [
    f"Feature_{i:03d}" for i in selected_indices
]

print("\nSelected Features:")
for feature in feature_names:
    print(feature)

selected_data = measurements[:, selected_indices]

feature_medians = np.nanmedian(
    selected_data,
    axis=0
)

imputed_data = np.where(
    np.isnan(selected_data),
    feature_medians,
    selected_data
)

BOOTSTRAP_REPETITIONS = 200 + (10 * SEED_DIGIT)

print("\nBootstrap Repetitions:", BOOTSTRAP_REPETITIONS)

def run_bootstrap(seed):

    rng = np.random.default_rng(seed)

    available_results = np.empty(
        (BOOTSTRAP_REPETITIONS, len(feature_names))
    )

    imputed_results = np.empty(
        (BOOTSTRAP_REPETITIONS, len(feature_names))
    )

    for repetition in range(BOOTSTRAP_REPETITIONS):

        pass_sample = rng.choice(
            pass_indices,
            size=len(pass_indices),
            replace=True
        )

        fail_sample = rng.choice(
            fail_indices,
            size=len(fail_indices),
            replace=True
        )

        for column, feature_index in enumerate(selected_indices):

            pass_values = measurements[
                pass_sample, feature_index
            ]

            fail_values = measurements[
                fail_sample, feature_index
            ]

            available_results[repetition, column] = (
                np.nanmean(fail_values)
                - np.nanmean(pass_values)
            )

            pass_values = imputed_data[
                pass_sample, column
            ]

            fail_values = imputed_data[
                fail_sample, column
            ]

            imputed_results[repetition, column] = (
                np.mean(fail_values)
                - np.mean(pass_values)
            )

    return available_results, imputed_results

available_bootstrap, imputed_bootstrap = run_bootstrap(
    RANDOM_SEED + 7
)

available_check, imputed_check = run_bootstrap(
    RANDOM_SEED + 7
)

reproducibility_verified = (
    np.array_equal(available_bootstrap, available_check)
    and np.array_equal(imputed_bootstrap, imputed_check)
)

comparison_rows = []

for column, feature_index in enumerate(selected_indices):

    pass_values = measurements[
        pass_indices, feature_index
    ]

    fail_values = measurements[
        fail_indices, feature_index
    ]

    available_difference = (
        np.nanmean(fail_values)
        - np.nanmean(pass_values)
    )

    pass_values = imputed_data[pass_indices, column]
    fail_values = imputed_data[fail_indices, column]

    imputed_difference = (
        np.mean(fail_values)
        - np.mean(pass_values)
    )

    available_ci = np.percentile(
        available_bootstrap[:, column],
        [2.5, 97.5]
    )

    imputed_ci = np.percentile(
        imputed_bootstrap[:, column],
        [2.5, 97.5]
    )

    comparison_rows.append({
        "Feature": feature_names[column],
        "Available_Case_Difference": available_difference,
        "Available_Case_CI_Lower": available_ci[0],
        "Available_Case_CI_Upper": available_ci[1],
        "Imputed_Difference": imputed_difference,
        "Imputed_CI_Lower": imputed_ci[0],
        "Imputed_CI_Upper": imputed_ci[1]
    })

comparison = pd.DataFrame(comparison_rows)

comparison["Available_Case_Rank"] = (
    comparison["Available_Case_Difference"]
    .abs()
    .rank(ascending=False, method="min")
    .astype(int)
)

comparison["Imputed_Rank"] = (
    comparison["Imputed_Difference"]
    .abs()
    .rank(ascending=False, method="min")
    .astype(int)
)

comparison["Direction_Matches"] = (
    np.sign(comparison["Available_Case_Difference"])
    == np.sign(comparison["Imputed_Difference"])
)

comparison["Rank_Matches"] = (
    comparison["Available_Case_Rank"]
    == comparison["Imputed_Rank"]
)

comparison["CI_Overlap"] = (
    (comparison["Available_Case_CI_Lower"]
     <= comparison["Imputed_CI_Upper"])
    &
    (comparison["Imputed_CI_Lower"]
     <= comparison["Available_Case_CI_Upper"])
)

comparison["Absolute_Change"] = (
    comparison["Available_Case_Difference"]
    - comparison["Imputed_Difference"]
).abs()

comparison = comparison.sort_values(
    "Available_Case_Rank"
)

comparison_path = OUTPUT_DIR / "robustness_comparison.csv"

comparison.to_csv(
    comparison_path,
    index=False
)

changed_sign = comparison[
    ~comparison["Direction_Matches"]
]["Feature"].tolist()

changed_ranking = comparison[
    ~comparison["Rank_Matches"]
]["Feature"].tolist()

largest_change = comparison["Absolute_Change"].max()

report_path = OUTPUT_DIR / "sensitivity_report.txt"

with open(report_path, "w", encoding="utf-8") as file:

    file.write("STATISTICAL ROBUSTNESS AND SENSITIVITY REPORT\n")
    file.write("============================================\n\n")

    file.write(f"Student: {LAST_NAME}\n")
    file.write(f"Student ID: {STUDENT_ID}\n")
    file.write(f"Random Seed: {RANDOM_SEED}\n")
    file.write(
        f"Bootstrap Repetitions: {BOOTSTRAP_REPETITIONS}\n\n"
    )

    file.write("SELECTED FEATURES\n")

    for feature in feature_names:
        file.write(f"- {feature}\n")

    file.write("\nCOMPARISON RESULTS\n")

    for _, row in comparison.iterrows():

        file.write(f"\nFeature: {row['Feature']}\n")

        file.write(
            f"Available-Case Difference: "
            f"{row['Available_Case_Difference']:.6f}\n"
        )

        file.write(
            f"Available-Case 95% CI: "
            f"{row['Available_Case_CI_Lower']:.6f} to "
            f"{row['Available_Case_CI_Upper']:.6f}\n"
        )

        file.write(
            f"Imputed Difference: "
            f"{row['Imputed_Difference']:.6f}\n"
        )

        file.write(
            f"Imputed 95% CI: "
            f"{row['Imputed_CI_Lower']:.6f} to "
            f"{row['Imputed_CI_Upper']:.6f}\n"
        )

        file.write(
            f"Available-Case Rank: "
            f"{row['Available_Case_Rank']}\n"
        )

        file.write(
            f"Imputed Rank: {row['Imputed_Rank']}\n"
        )

    file.write("\nSENSITIVITY SUMMARY\n")

    file.write(
        f"Largest Change in Mean Difference: "
        f"{largest_change:.6f}\n"
    )

    file.write(
        f"Features with Changed Difference Sign: "
        f"{changed_sign}\n"
    )

    file.write(
        f"Features with Changed Ranking: "
        f"{changed_ranking}\n"
    )

    file.write(
        f"Reproducibility Verified: "
        f"{reproducibility_verified}\n"
    )

print("\nROBUSTNESS COMPARISON")
print(comparison.round(6).to_string(index=False))

print("\nASSESSMENT DATA")
print("Selected Features:", len(feature_names))
print("Bootstrap Repetitions:", BOOTSTRAP_REPETITIONS)
print("Largest Change in Mean Difference:", round(largest_change, 6))
print("Features with Changed Difference Sign:", changed_sign)
print("Features with Changed Ranking:", changed_ranking)
print("Reproducibility Verified:", reproducibility_verified)

print("\nGENERATED FILES")
print("Comparison CSV:", comparison_path.exists())
print("Sensitivity Report:", report_path.exists())

=== EXERCISE 3: ROBUSTNESS AND SENSITIVITY ===
Student: German
Student ID: TUPM-26-4336
Random Seed: 264336

Selected Features:
Feature_054
Feature_118
Feature_250
Feature_351
Feature_523

Bootstrap Repetitions: 260

ROBUSTNESS COMPARISON
    Feature  Available_Case_Difference  Available_Case_CI_Lower  Available_Case_CI_Upper  Imputed_Difference  Imputed_CI_Lower  Imputed_CI_Upper  Available_Case_Rank  Imputed_Rank  Direction_Matches  Rank_Matches  CI_Overlap  Absolute_Change
Feature_250                   1.526971                -5.816813                 9.495886            1.526971         -5.816813          9.495886                    1             1               True          True        True         0.000000
Feature_523                  -0.173253                -0.426452                 0.073885           -0.173253         -0.426452          0.073885                    2             2               True          True        True         0.000000
Feature_351                   0.004